In [ ]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# ============================================================
# CONFIGURATION
# ============================================================
# Paths to the label CSV, masked image dataset, model checkpoint and plot output directory
CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/boneage-training-dataset.csv"
IMAGE_DIR = "/content/drive/MyDrive/Colab Notebooks/overlayed_RSNA_dataset"
SAVE_PATH = "/content/drive/MyDrive/Colab Notebooks/resnet50_best.pth"
PLOT_DIR = "/content/drive/MyDrive/Colab Notebooks/training_plots_resnet50"

# Training hyperparameters
IMAGE_SIZE = 448
BATCH_SIZE = 32
NUM_EPOCHS = 100
LR = 1e-4
WEIGHT_DECAY = 1e-4
VAL_SIZE = 0.2
RANDOM_SEED = 42
NUM_WORKERS = 2
EARLY_STOPPING_PATIENCE = 10

# Use GPU if available, otherwise fall back to CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Accepted image file types
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

# Enables faster convolution algorithms for fixed input sizes
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True


def set_seed(seed=42):
    """
    Set random seeds to make the train/validation split and training behavior
    more reproducible.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_SEED)


def get_transforms():
    """
    Define preprocessing for ResNet50.

    The masked X-ray images are resized to 448x448, converted to tensors,
    and normalized with ImageNet mean/std because the ResNet50 backbone
    was pretrained on ImageNet.
    """
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])


class BoneAgeDataset(Dataset):
    """
    Custom PyTorch Dataset for bone age prediction.

    Each sample returns:
    - image tensor
    - gender tensor
    - bone age target tensor
    """
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Image ID must match the filename in IMAGE_DIR
        image_id = str(row["id"])

        # Bone age is the regression target, measured in months
        boneage = float(row["boneage"])

        # Convert gender to numeric format:
        # male = 1.0, female = 0.0
        male = float(row["male"].strip().lower() == "true" if isinstance(row["male"], str) else row["male"])

        # Locate image file and load it as RGB
        # RGB is used because ImageNet-pretrained ResNet expects 3-channel input
        image_path = self._find_file(self.image_dir, image_id)
        image = Image.open(image_path).convert("RGB")

        # Apply resizing, tensor conversion and normalization
        if self.transform:
            image = self.transform(image)

        # Gender is added as an additional numerical feature
        gender = torch.tensor([male], dtype=torch.float32)

        # Target is wrapped in a one-value tensor for regression
        target = torch.tensor([boneage], dtype=torch.float32)

        return image, gender, target

    @staticmethod
    def _find_file(folder, image_id):
        """
        Search for an image file with one of the accepted extensions.
        """
        for ext in [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]:
            path = os.path.join(folder, image_id + ext)
            if os.path.exists(path):
                return path
        raise FileNotFoundError(f"No file found for ID {image_id}")


class BoneAgeResNetWithGender(nn.Module):
    """
    ResNet50-based regression model with gender metadata.

    The CNN extracts image features.
    The gender value is concatenated to these image features.
    A small regression head predicts bone age in months.
    """
    def __init__(self):
        super().__init__()

        # Load ImageNet-pretrained ResNet50 backbone
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

        # Store number of extracted image features before replacing final layer
        num_features = backbone.fc.in_features

        # Remove original ImageNet classification layer
        backbone.fc = nn.Identity()
        self.backbone = backbone

        # Regression head:
        # num_features image features + 1 gender feature -> bone age prediction
        self.regressor = nn.Sequential(
            nn.Linear(num_features + 1, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )

    def forward(self, images, gender):
        # Extract image features from ResNet50
        image_features = self.backbone(images)

        # Combine image features with gender metadata
        combined = torch.cat([image_features, gender], dim=1)

        # Predict bone age
        return self.regressor(combined)


def train_one_epoch(model, loader, criterion, optimizer, device):
    """
    Train the model for one epoch.
    Returns the average training loss.
    """
    model.train()
    running_loss = 0.0

    for images, gender, targets in tqdm(loader, desc="Training", leave=False):
        images, gender, targets = images.to(device), gender.to(device), targets.to(device)

        # Reset gradients from previous batch
        optimizer.zero_grad()

        # Forward pass and loss calculation
        loss = criterion(model(images, gender), targets)

        # Backpropagation and parameter update
        loss.backward()
        optimizer.step()

        # Accumulate loss weighted by batch size
        running_loss += loss.item() * images.size(0)

    return running_loss / len(loader.dataset)


@torch.no_grad()
def validate(model, loader, criterion, device):
    """
    Evaluate the model on the validation set.
    Returns validation loss, validation MAE, predictions and targets.
    """
    model.eval()
    running_loss = 0.0
    preds_all, targets_all = [], []

    for images, gender, targets in tqdm(loader, desc="Validation", leave=False):
        images, gender, targets = images.to(device), gender.to(device), targets.to(device)

        outputs = model(images, gender)

        # Validation loss
        running_loss += criterion(outputs, targets).item() * images.size(0)

        # Store predictions and targets for MAE and scatter plot
        preds_all.append(outputs.cpu().numpy())
        targets_all.append(targets.cpu().numpy())

    preds_all, targets_all = np.concatenate(preds_all).ravel(), np.concatenate(targets_all).ravel()

    # MAE is measured in months because the target boneage is in months
    return running_loss / len(loader.dataset), np.mean(np.abs(preds_all - targets_all)), preds_all, targets_all


def save_training_plots(train_losses, val_losses, val_maes, y_true, y_pred, output_dir):
    """
    Save a 2x2 training summary figure:
    - Training loss
    - Validation loss
    - Validation MAE
    - True vs predicted bone age scatter plot
    """
    os.makedirs(output_dir, exist_ok=True)

    epochs = range(1, len(train_losses) + 1)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].plot(epochs, train_losses)
    axes[0, 0].set_title("Train Loss")

    axes[0, 1].plot(epochs, val_losses)
    axes[0, 1].set_title("Val Loss")

    axes[1, 0].plot(epochs, val_maes)
    axes[1, 0].set_title("Val MAE")

    # Scatter plot: ideal predictions would lie on the red diagonal line
    axes[1, 1].scatter(y_true, y_pred, alpha=0.5)
    axes[1, 1].plot([min(y_true), max(y_true)], [min(y_true), max(y_true)], 'r--')

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "training_summary.png"))
    plt.show()


def main():
    """
    Full training pipeline:
    1. Load labels
    2. Keep only rows with available masked images
    3. Split data into train and validation sets
    4. Train ResNet50 regression model
    5. Save best model based on validation MAE
    6. Save final training plots
    """
    os.makedirs(PLOT_DIR, exist_ok=True)

    # Load CSV with id, boneage and gender
    df = pd.read_csv(CSV_PATH)

    # Keep only labels for which a masked image actually exists
    available_ids = {
        os.path.splitext(f)[0]
        for f in os.listdir(IMAGE_DIR)
        if os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS
    }

    df = df[df["id"].astype(str).isin(available_ids)].reset_index(drop=True)

    # Random train/validation split
    train_df, val_df = train_test_split(df, test_size=VAL_SIZE, random_state=RANDOM_SEED)

    # DataLoaders handle batching, shuffling and parallel loading
    train_loader = DataLoader(
        BoneAgeDataset(train_df, IMAGE_DIR, get_transforms()),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    val_loader = DataLoader(
        BoneAgeDataset(val_df, IMAGE_DIR, get_transforms()),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    # Initialize model, loss function and optimizer
    model = BoneAgeResNetWithGender().to(DEVICE)

    # L1Loss corresponds directly to MAE optimization
    criterion = nn.L1Loss()

    # AdamW with weight decay for regularization
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # Reduces learning rate when validation MAE stops improving
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=3
    )

    best_val_mae = float("inf")
    epochs_no_improve = 0
    train_losses, val_losses, val_maes = [], [], []

    for epoch in range(NUM_EPOCHS):
        # Train and validate one epoch
        t_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        v_loss, v_mae, p, t = validate(model, val_loader, criterion, DEVICE)

        train_losses.append(t_loss)
        val_losses.append(v_loss)
        val_maes.append(v_mae)

        print(
            f"Epoch {epoch+1} | "
            f"Train Loss: {t_loss:.4f} | "
            f"Val MAE: {v_mae:.4f} | "
            f"LR: {optimizer.param_groups[0]['lr']:.6f}"
        )

        # Update learning rate based on validation MAE
        scheduler.step(v_mae)

        # Save best model checkpoint based on validation MAE
        if v_mae < best_val_mae:
            best_val_mae = v_mae
            epochs_no_improve = 0
            torch.save(model.state_dict(), SAVE_PATH)
            best_p, best_t = p, t
            print("Best model saved.")
        else:
            epochs_no_improve += 1

        # Stop training if validation MAE does not improve for several epochs
        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break

    # Save plots using predictions from the best validation epoch
    save_training_plots(train_losses, val_losses, val_maes, best_t, best_p, PLOT_DIR)


if __name__ == "__main__":
    main()

Using device: cuda
Image size: 448x448
Batch size: 32
Loss function: smooth_l1


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/boneage-training-dataset.csv'

In [ ]:
import os

print(os.listdir('/content/drive/MyDrive/Colab Notebooks'))
print(os.cpu_count())

['training_plots_resnet50_448']
2


In [ ]:
!nvidia-smi